# Part 1. Data Processing
We will be using the ```pandas``` library to represent our data.

# 1.1. Cleaning
To get an initial overview of the data, we will load and preprocess a 250-row sample.

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

tqdm.pandas()
np.random.seed(42) # Seeding for reproducibility

df = pd.read_csv("data/news_sample.csv", index_col=0)
df.head(3)

,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary
0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,Sometimes the power of Christmas will make you...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Church Congregation Brings Gift to Waitresses ...,Ruth Harris,NaN,[''],NaN,NaN,NaN
1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,Zurich Times,NaN,[''],NaN,NaN,NaN
2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,Never Hike Alone: A Friday the 13th Fan Film U...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Never Hike Alone - A Friday the 13th Fan Film ...,NaN,NaN,[''],Never Hike Alone: A Friday the 13th Fan Film ...,NaN,NaN


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 249
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                250 non-null    int64  
 1   domain            250 non-null    object 
 2   type              238 non-null    object 
 3   url               250 non-null    object 
 4   content           250 non-null    object 
 5   scraped_at        250 non-null    object 
 6   inserted_at       250 non-null    object 
 7   updated_at        250 non-null    object 
 8   title             250 non-null    object 
 9   authors           170 non-null    object 
 10  keywords          0 non-null      float64
 11  meta_keywords     250 non-null    object 
 12  meta_description  54 non-null     object 
 13  tags              27 non-null     object 
 14  summary           0 non-null      float64
dtypes: float64(2), int64(1), object(12)
memory usage: 31.2+ KB


In [3]:
df["type"].unique()

array(['unreliable', 'fake', 'clickbait', 'conspiracy', 'reliable',
       'bias', 'hate', 'junksci', 'political', nan, 'unknown'],
      dtype=object)

In [4]:
print(f"Percentage of type rows null: {100 - (df['type'].notna().sum() / len(df['type'])*100):.2f}%")

Percentage of type rows null: 4.80%


From this we can already make the following observations about the different columns:
1. Id: We need to verify this, but this is probably a unique identifier for the row.
2. Domain: The domain from which the article is scraped.
3. Type: The actual label for the reliability of the article.
4. Url: The link to the article.
5. Content: The actual content of the article.
6. scraped_at, inserted_at, updated_at: Timestamps recording certain alterations of the data.
7. Title: The title of the article.
8. Authors: The list of authors for the article
9. Keywords: All null.
10. Meta-keywords: We need to analyze this column further to distinguish it from keywords.
11. Meta-description: Description of the article.
12. Tags: Probably the same as keywords.
13. Summary: All null

Now we have the following tasks:
1. Drop uninformative columns. These are:
    - scraped_at
    - inserted_at
    - updated_ad
    - authors: We remove authors since a name itself doesn't imply anything about the content's reliability. The only thing a model would learn would be that certain authors are less reliable than others.
    - keywords, tags and meta_keywords: It seems that only a subset of rows have either of these, which will risk biasing modelling.
    - meta_description and summary: Same argument as above.
2. Deduplicate content. For modelling, we do not want duplicates since they will bias the model.
3. Remove rows where either type or content is missing. If it only a handful of our data points was missing a type, we could probably manually impute the type value, however given that ~5% of our data has a missing value in the type column, we cannot reliably impute the values, and for 1M rows, this would mean we have ~50.000 rows of missing values.
4. Remove "unknown" type values.
5. Cast the non-null type values into a ```pandas``` category type.
6. Drop rows that are ill-formed and do not have a proper numerical vlaue in the ```id``` column.

**Note:** Given that each column has a unique identifier (```Id```), we can safely drop columns, since we can always get the data from the raw dataset.

In [5]:
from sklearn_transformers import FilterTransformer

filter_tr = FilterTransformer(
    drop_cols=["scraped_at", "inserted_at", "updated_at", "authors", "keywords", "tags", "meta_keywords", "meta_description", "summary"],
    remove_nulls_col_names=["type", "content"],
    deduplicate_cols=["content"],
    convert_to_category_cols=["type"],
    remove_cols_with_value={"type":"unknown"})

filtered_df = filter_tr.transform(df)
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221 entries, 0 to 246
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   id       221 non-null    int64   
 1   domain   221 non-null    object  
 2   type     221 non-null    category
 3   url      221 non-null    object  
 4   content  221 non-null    object  
 5   title    221 non-null    object  
dtypes: category(1), int64(1), object(4)
memory usage: 10.9+ KB


Now that our dataset is in some kind of shape for further processing, we can begin cleaning it more thoroughly. Given that we're working with natural language, we can expect to encounter multiple types of elements that are not directly relevant to the meaning of the given text. This includes:
1. Non-NLP entities: We will be performing masking on e-mails, dates, URl's and numbers, however this is not exhaustive. Other entities could include adresses, names, locations etc..
2. Punctuation: It is debatable whether punctuation directly contributes to a document's semantic meaning, but for initial cleaning purposes, we will keep it.
3. Casing: In most cases, we can expect words to have the same meaning regardless of the casing of their characters. There are exceptions though, but due to limited compute resources, we will favour a smaller vocabulary over 100% precision.

We will be cleaning the texts using our own date-matching regular expression and the ```clean-text``` library. Additionally, the only special characters we will remove are ```<``` and ```>``` since these will be used for our masking tokens.

Let's sample a document before and after cleaning:

In [6]:
sample = filtered_df.iloc[1].content
print(sample)

AWAKENING OF 12 STRANDS of DNA – “Reconnecting with You” Movie

% of readers think this story is Fact. Add your two cents.

Headline: Bitcoin & Blockchain Searches Exceed Trump! Blockchain Stocks Are Next!

[January 24, 2018 - ZurichTimes.net]

As Miles Johnston was giving update, it was another case of Strange Synchronicities of Goodness Hidden inside of Tests and Trials, like a Follow the WhiteRabbit down the Rabbit Hole type of exercise.

In Researching the 12 Strands of DNA we came across some articles, one in particular was as a Strange Synchronicity written exactly 1 year ago on the Same Topic.

https://www.youtube.com/watch?v=_6Ze1mrs4BQ

https://www.youtube.com/watch?v=2nKcDiIc8JY

What are the 12 Strands of our DNA and Why is a War Against our DNA?

Trailer for Awakening of 12 Strands

The Full Video is only available as a Paid Video on Vimeo.

AWAKENING OF 12 STRANDS – “Reconnecting with You”

vimeo.com/ondemand/Awakeningof12strands

AWAKENING OF 12 STRANDS – “Reconnecting wi

In [7]:
from sklearn.pipeline import Pipeline
from sklearn_transformers import CleaningTransformer
import warnings

warnings.filterwarnings(
    "ignore",
    message="This Pipeline instance is not fitted yet.*",
    category=FutureWarning,
)

clean_pl = Pipeline([
    ("filter", filter_tr),
    ("clean", CleaningTransformer())])
clean_pl.fit(filtered_df)
cleaned = clean_pl.transform(filtered_df)
cleaned.iloc[1].content

'awakening of <NUM> strands of dna – “reconnecting with you” movie % of readers think this story is fact. add your two cents. headline: bitcoin & blockchain searches exceed trump! blockchain stocks are next! [<DATE> - zurichtimes.net] as miles johnston was giving update, it was another case of strange synchronicities of goodness hidden inside of tests and trials, like a follow the whiterabbit down the rabbit hole type of exercise. in researching the <NUM> strands of dna we came across some articles, one in particular was as a strange synchronicity written exactly <NUM> year ago on the same topic. <URL> <URL> what are the <NUM> strands of our dna and why is a war against our dna? trailer for awakening of <NUM> strands the full video is only available as a paid video on vimeo. awakening of <NUM> strands – “reconnecting with you” vimeo.com/ondemand/awakeningof<NUM>strands awakening of <NUM> strands – “reconnecting with you”. from sandra daroy on vimeo. we have not watched the full video, 

In [8]:
cleaned.head()

,id,domain,type,url,content,title
0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,sometimes the power of christmas will make you...,church congregation brings gift to waitresses ...
1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,awakening of <NUM> strands of dna – “reconnect...,awakening of <NUM> strands of dna – “reconnect...
2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,never hike alone: a friday the <NUM>th fan fil...,never hike alone - a friday the <NUM>th fan fi...
3,768,awm.com,unreliable,http://awm.com/elusive-alien-of-the-sea-caught...,"when a rare shark was caught, scientists were ...",elusive ‘alien of the sea ‘ caught by scientis...
4,791,bipartisanreport.com,clickbait,http://bipartisanreport.com/2018/01/21/trumps-...,donald trump has the unnerving ability to abil...,trump’s genius poll is complete & the results ...


## 1.2. Tokenization
For the tokenization step, we remove all special characters but ```<>``` and split on whitespace. Then we perform stopwords removal as well as simple stemming.

Additionally, we provide our tokenizer with a list of special-tokens (```<NUM>```, ```<DATE>``` etc.) to ensure that they are included in the vocabulary, even if its size is limited.

To argue for the two latter methods's effectiveness, we will now be demonstrating vocabulary shrinkage when they are employed.

In [9]:
from sklearn_transformers import TokenTransformer
from nltk.corpus import stopwords
import Stemmer
import nltk

nltk.download('stopwords', quiet=True)
SPECIAL_TOKENS = ["<NUM>", "<DATE>", "<EMAIL>", "<URL>"]

raw_vocab = TokenTransformer(top_n=None, special_tokens=SPECIAL_TOKENS, stopwords=[], stem=False).fit(cleaned)
vocab_no_sw = TokenTransformer(top_n=None, special_tokens=SPECIAL_TOKENS, stopwords=stopwords.words("english"), stem=False).fit(cleaned)
vocab_no_sw_snow = TokenTransformer(top_n=None, special_tokens=SPECIAL_TOKENS, stopwords=stopwords.words("english"), stem=True).fit(cleaned)

reduction_df = pd.DataFrame({
    "Method": [
        "Raw",
        "SW removal",
        "SW removal + SnowballStemmer"
    ],
    "Vocab Size": [raw_vocab.size(), vocab_no_sw.size(), vocab_no_sw_snow.size()],
})
reduction_df["Reduction vs. Raw"] = raw_vocab.size() - reduction_df["Vocab Size"]
reduction_df["Reduction rate"] = (reduction_df["Reduction vs. Raw"] / raw_vocab.size()) * 100
reduction_df

100%|██████████| 221/221 [00:00<00:00, 2279.13it/s]


,Method,Vocab Size,Reduction vs. Raw,Reduction rate
0,Raw,15598,0,0.000000
1,SW removal,15466,132,0.846262
2,SW removal + SnowballStemmer,10351,5247,33.638928


As we can see, there is a significant decrease in vocabulary size when stemming is applied. However, we can also see from the running times, that introducing stemming also slows down the algorithm by ~30%.

Despite this we deem the decrease in sparsity to outweigh any speed decreases.

# 2. Expanding the dataset

## 2.1. Cleaning

In [10]:
from pathlib import Path
import os

CHUNK_SIZE = 2048
VOCABULARY_SIZE = 10000
DATA_DIR = Path("data")
FINAL_DS_PATH = DATA_DIR / "preprocessed"
TOKENIZERS_DIR = FINAL_DS_PATH / "tokenizers"
SPLITS_PATH = FINAL_DS_PATH / "splits"

CLEANED_DS_PATH = FINAL_DS_PATH / "995k_rows_cleaned.csv"
FULL_DATASET = DATA_DIR / "995,000_rows.csv"
if not os.path.isdir(FINAL_DS_PATH):
    os.mkdir(FINAL_DS_PATH)

if not os.path.isdir(TOKENIZERS_DIR):
    os.mkdir(TOKENIZERS_DIR)
if not os.path.isdir(SPLITS_PATH):
    os.mkdir(SPLITS_PATH)

row_count = len(pd.read_csv(FULL_DATASET, usecols=["id"]))
print(f"Reading in dataset with {row_count} rows.")
large_df = pd.read_csv(
    FULL_DATASET,  
    usecols=["id", "domain", "type", "url", "content", "title"],
    chunksize=CHUNK_SIZE)
large_df

Reading in dataset with 995000 rows.


C:\Users\nva\AppData\Local\Temp\ipykernel_20280\654228068.py:21: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  row_count = len(pd.read_csv(FULL_DATASET, usecols=["id"]))


In [11]:
clean_pipeline = Pipeline([
    ("filter", FilterTransformer(
        drop_cols=["scraped_at", "inserted_at", "updated_at", "authors", "keywords", "tags", "meta_keywords", "meta_description", "summary"],
        remove_nulls_col_names=["type", "content"],
        deduplicate_cols=["content"],
        convert_to_category_cols=["type"],
        remove_cols_with_value={"type":"unknown"})),
    ("clean", CleaningTransformer())
])

clean_pipeline.fit(df) # Avoid NotFittedError
first_chunk = True

chunk_count = (row_count // CHUNK_SIZE) + 1
print(f"Cleaning dataset in {chunk_count} chunks\n")
for chunk in tqdm(large_df, total=chunk_count):
    cleaned = clean_pipeline.transform(chunk)
    cleaned.to_csv(
        CLEANED_DS_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False)
    first_chunk = False

Cleaning dataset in 486 chunks



100%|██████████| 486/486 [01:33<00:00,  5.18it/s]


In [12]:
from joblib import dump
dump(clean_pipeline, FINAL_DS_PATH / "clean_pipeline.joblib")

['data\\preprocessed\\clean_pipeline.joblib']

## 2.2. Train / Test / Validation split
We will be splitting the dataset into 80% training data and 10% test and validation data respectively. To do so, we will be using scikit-learn to randomly split the dataset.

In [13]:
cleaned_df = pd.read_csv(CLEANED_DS_PATH)
cleaned_df.head(3)

,id,domain,type,url,content,title
0,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,plus one article on google plus (thanks to ali...,iran news round up
1,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,the cost of the best senate banking committee ...,the cost of the best senate banking committee ...
2,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,man awoken from <NUM>-year coma commits suicid...,man awoken from <NUM>-year coma commits suicid...


In [14]:
from sklearn.model_selection import train_test_split

# TODO: A seed has been added for reproducibility. If you wish total randomness, set this to None
RNG_SEED = 1234
train_df, test_df = train_test_split(cleaned_df, test_size=0.2, random_state=RNG_SEED)
test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=RNG_SEED)

print("Length of train dataset: ", len(train_df))
print("Length of test dataset: ", len(test_df))
print("Length of validation dataset: ", len(val_df))

Length of train dataset:  634696
Length of test dataset:  79337
Length of validation dataset:  79337


In [15]:
train_df.to_csv(SPLITS_PATH / "train.csv", index=False)
print("Saved training dataset")

test_df.to_csv(SPLITS_PATH / "test.csv", index=False)
print("Saved testing dataset")

val_df.to_csv(SPLITS_PATH / "val.csv", index=False)
print("Saved validation dataset")

Saved training dataset
Saved testing dataset
Saved validation dataset


## 2.3. Tokenization
To avoid data leakage, we will only be fitting our tokenizers on the training data.

In [16]:
tokenizer = TokenTransformer(
    top_n=VOCABULARY_SIZE, 
    special_tokens=SPECIAL_TOKENS,
      stopwords=stopwords.words("english"), 
      stem=True
    ).fit(train_df)

print("Vocabulary size: ", tokenizer.size())
dump(tokenizer, TOKENIZERS_DIR / "top10k.joblib")

100%|██████████| 634696/634696 [03:09<00:00, 3355.64it/s]


Vocabulary size:  10000


['data\\preprocessed\\tokenizers\\top10k.joblib']

For later analysis we will also compute the vocabulary without stemming, as well as without both stemming and stopwords:

In [17]:
tokenizer = TokenTransformer(
    top_n=VOCABULARY_SIZE, 
    special_tokens=SPECIAL_TOKENS,
      stopwords=stopwords.words("english"), 
      stem=False
    ).fit(train_df)

print("Vocabulary size without stemming: ", tokenizer.size())
dump(tokenizer, TOKENIZERS_DIR / "top10k_nostem.joblib")

100%|██████████| 634696/634696 [01:50<00:00, 5746.43it/s]


Vocabulary size without stemming:  10000


['data\\preprocessed\\tokenizers\\top10k_nostem.joblib']

In [18]:
tokenizer = TokenTransformer(
    top_n=VOCABULARY_SIZE, 
    special_tokens=SPECIAL_TOKENS,
      stopwords=[], 
      stem=False
    ).fit(train_df)

print("Vocabulary size without stemming and stopword removal: ", tokenizer.size())
dump(tokenizer, TOKENIZERS_DIR / "top10k_nostem_nosw.joblib")

  0%|          | 0/634696 [00:00<?, ?it/s]

100%|██████████| 634696/634696 [01:52<00:00, 5643.49it/s]


Vocabulary size without stemming and stopword removal:  10000


['data\\preprocessed\\tokenizers\\top10k_nostem_nosw.joblib']